# Rapor: Tokenizer Sensitivity

Bu çalışma, aynı anlama sahip görünsel olarak benzer metinlerin tokenizer tarafından nasıl farklı parçalandığını ve model çıktısına etkisini göstermeyi amaçlamaktadır. Özellikle görsel benzerlik, görünmez karakterler, benzer harfler ve küçük yazım hataları modeli nasıl yönlendirebilir, incelenmiştir.

## Amaç
- Tokenizer, aynı kelimeyi farklı yazım formlarında aynı şekilde mi algılar?
- Görsel olarak benzer ama farklı Unicode karakterler içeren metinler nasıl işlenir?
- Çift yönlü görsel manipülasyon ve zero-width karakterler model davranışını etkiler mi?

## Gözlem
Model, metnin semantik anlamına göre değil, tokenize edilmiş temsiline göre karar verir. Bu nedenle:
- Kiril alfabesiyle yazılmış benzer harfler, Latin harflerine göre farklı token üretimi oluşturabilir.
- Metin içindeki görünmez boşluklar veya noktalama benzeri karakterler tokenizer için farklı kesimlere neden olabilir.
- Kısmi yazım hataları, kaynaştırılmış karakterler ve görsel olarak benzer varyasyonlar modelin güvenini bozabilir.

## Önemli Nokta
Bir cümlenin insan için açık ve anlaşılır olması, tokenizer için aynı zaman da güvenli bir temsil üretmesi gerekmez. Bu durum, özellikle duygu analizi, sınıflandırma ve güvenlik testleri açısından ciddi bir risk oluşturur.

## Sonuç
Tokenizer sensitivity, modelin karar mekanizmasının düşük seviye karakter ve token ayrımlarına bağımlı hale gelmesine sebep olur. Bu tür varyasyonlar, sistemin semantik dayanıklılığını azaltır ve güvenli kullanım için ek denetim gerektirir.

## Değerlendirme
- Testlerde yalnızca metinlerin anlamı değil, karakter düzeyindeki varyasyonları da incelemek gerekir.
- Tokenizer doğrulama adımları, Unicode ve zero-width karakter kontrolü ile desteklenmelidir.
- Model çıktıları okunabilirlik ve güven aralığı açısından birlikte değerlendirilmelidir.

Bu rapor, dili temsil eden sembollerin bile model davranışında önemli etki yaratabileceğini göstermektedir.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Model checkpoints
model_names = {
    "DistilBERT": "distilbert-base-uncased-finetuned-sst-2-english",
    "RoBERTa": "textattack/roberta-base-SST-2",
    "ALBERT": "textattack/albert-base-v2-SST-2"
}

# 1. PRE-LOAD EVERYTHING FIRST
print("Loading all models and tokenizers into memory...")
loaded_models = {}

for name, checkpoint in model_names.items():
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
    loaded_models[name] = (tokenizer, model)

print("All models loaded successfully!\n")

Loading all models and tokenizers into memory...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

...

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 46.7MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

All models loaded successfully!



In [ ]:

print("=== TOKENIZER SENSIVITY ===")

texts = [
    "This movie was terrible. I hated it.",
    "This movie was ter rible. I h ated it.",
    "This movie was t.e.r.r.i.b.l.e. I h.a.t.e.d it.",
    "This movie was t*rr*ble. I h*ted it.",
    "This movie was terr*ble. I hat*d it.",
    # Typoglycemia Test
    "This mvoie was absloutely trrible. I haetd it.",
    # 1. Homoglif Saldırısı (Sadece Kiril 'e' [\u0435] ve Kiril 'a' [\u0430] kullanımı)
    # "terrible" ve "hated" içindeki 'e' ve 'a' harfleri Kiril alfabesindendir.
    "This movie was tеrriblе. I hаtеd it.",

    # 2. Tam Görsel Manipülasyon (Kiril 'o', 'a', 'e', 'i' [\u043e, \u0430, \u0435, \u0456])
    # E-posta spam filtrelerini veya model duygu analizini atlatmak için kullanılan yöntem.
    "Thіs mоvіe wаs tеrrіblе. I hаtеd іt.",

    # 3. Görünmez Karakter Enjeksiyonu (Zero-Width Space [\u200b])
    # Kelimelerin arasına görünmez boşluk konur, insan kelimeyi tek parça görür ama tokenizer böler.
    "This movie was ter\u200brible. I hat\u200bed it.",
    "I hate it!",
    "the acting is amateurish",
    "the acting is amateirish",
    "the acting is aateurish",
    "the acting is enthusiastic",

    """
    I

    h
    a
    t
    e

    i
    t
    !
    """,
    "This film was filmed in France",
    "This film was filmed in Afghanistan"
]

for name, (tokenizer, model) in loaded_models.items():
    model.eval()
    print("\n\nMODEL:", name)

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")

        with torch.no_grad():
            output = model(**inputs).logits

        probs = torch.softmax(output, dim=1)[0]

        probs_label = "Positive" if probs.argmax().item() else "Negative"

        print(
            text,
            "| Sinif:", probs_label,
            "| Guven:", round(probs.max().item() * 100, 2), "%"
        )

=== TOKENIZER SENSIVITY ===


MODEL: DistilBERT
This movie was terrible. I hated it. | Sinif: Negative | Guven: 99.97 %
This movie was ter rible. I h ated it. | Sinif: Negative | Guven: 98.38 %
This movie was t.e.r.r.i.b.l.e. I h.a.t.e.d it. | Sinif: Negative | Guven: 99.2 %
This movie was t*rr*ble. I h*ted it. | Sinif: Negative | Guven: 99.22 %
This movie was terr*ble. I hat*d it. | Sinif: Negative | Guven: 54.78 %
This mvoie was absloutely trrible. I haetd it. | Sinif: Negative | Guven: 99.39 %
This movie was tеrriblе. I hаtеd it. | Sinif: Negative | Guven: 99.13 %
Thіs mоvіe wаs tеrrіblе. I hаtеd іt. | Sinif: Negative | Guven: 96.0 %
This movie was ter​rible. I hat​ed it. | Sinif: Negative | Guven: 99.97 %
I hate it! | Sinif: Negative | Guven: 99.96 %
the acting is amateurish | Sinif: Negative | Guven: 99.98 %
the acting is amateirish | Sinif: Positive | Guven: 97.25 %
the acting is aateurish | Sinif: Positive | Guven: 80.14 %
the acting is enthusiastic | Sinif: Positive | Guven: 99